# Сравнение OCR: новая модель (HTR) vs старая (VLM)

Новая модель распознаёт **одну картинку → строки текста** (несколько страниц на одной картинке она не выделяет), поэтому сохраняем **по одному файлу на картинку**: `OUT_DIR/<ДАТА>_<картинка>.json`, содержимое `{"id","text"}` как у старой модели.

Старая модель (VLM) хранила по нескольку картинок в одном json (блоки `Изображение 1:`, `Изображение 2:`… — по числу поданных за запрос картинок). Эти сочетания к нашей модели неприменимы, поэтому для сравнения берём только **одиночные** старые json.

Ноутбук: прогон новой модели (строки + полигоны детектора, с кэшем) → сохранение по файлу на картинку → сводка → **HTML-отчёт** `REPORT_FILE`: для каждой картинки оригинал + детекция детектора (пронумерованные боксы) и пословный дифф old↔new. Отчёт самодостаточный (картинки вшиты в base64), открывается в браузере. CER/WER — мера расхождения, не ground truth.

> Если строки уже были в кэше, а полигонов ещё нет — прогон **до-запросит** сервис, чтобы получить `coords` (иначе оверлей будет пустой).

In [ ]:
# --- настройки (поменяй под машину, где крутится клиент новой модели) ---
from datetime import date

IMAGES_DIR  = r"C:\Users\dim\Downloads\Images\Images"          # исходные картинки
OCR_DIR     = r"C:\Users\dim\Downloads\resultOCR2\resultOCR2"   # json старой модели (VLM)
OUT_DIR     = "resultOCR_new"          # сюда пишем новую модель: по одному файлу на картинку
ENDPOINT    = "https://entrypoint-htr.internal.k8s.antiplagiat-team.ru/"
CACHE_FILE  = "new_preds_cache.json"   # кэш строк по картинкам
COORDS_FILE = "new_coords_cache.json"  # кэш полигонов детектора по картинкам
REPORT_FILE = "ocr_report.html"        # самодостаточный HTML-отчёт (картинки вшиты в base64)
DATE        = date.today().isoformat() # префикс имени: <DATE>_<картинка>.json (поставь "2026-04-08" чтобы совпало со старыми)
MAX_IMG_WIDTH = 700                    # ширина превью картинки, px

In [ ]:
import os, re, json, html, uuid, difflib
from IPython.display import display, HTML, Image


def nat_key(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", str(s))]


def strip_prefix(text):
    # маркеры "Изображение N:" убираем и у старой, и у новой — дифф по тексту, не по меткам
    return re.sub(r"Изображение\s*\d+\s*:\s*", "", text).strip()


def load_text(json_path):
    # strict=False: json'ы старой модели содержат сырые \n внутри строк (невалидный JSON)
    with open(json_path, encoding="utf-8") as f:
        return json.load(f, strict=False).get("text", "")


def norm(t):
    return re.sub(r"\s+", " ", t).strip()


# имена реальных картинок — по ним разбираем склеенные имена json'ов (сначала длинные, чтобы image_2 не съел image_21)
known_images = sorted(
    [n for n in os.listdir(IMAGES_DIR) if n.lower().endswith((".png", ".jpg", ".jpeg"))],
    key=len, reverse=True,
)


def parse_doc(json_name):
    # "2026-04-08_image_2.png_image_21.jpg.json" -> ["image_2.png", "image_21.jpg"]
    m = re.match(r"(\d{4}-\d{2}-\d{2})_(.+)\.json$", json_name)
    if not m:
        return None
    body, i, parts = m.group(2), 0, []
    while i < len(body):
        hit = next((img for img in known_images if body[i:].startswith(img)), None)
        if not hit:
            return None
        parts.append(hit)
        i += len(hit)
        if i < len(body) and body[i] == "_":
            i += 1
    return parts


def _lev(a, b):
    # расстояние Левенштейна между последовательностями (симметрично: свап для скорости)
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def metrics(ref, hyp):
    # CER/WER без jiwer; делим на длину reference (старой модели)
    r, h = norm(ref), norm(hyp)

    def rate(a, b):
        if len(a) == 0:
            return 0.0 if len(b) == 0 else float("nan")  # пусто↔пусто = 0, есть↔пусто у ref = не определено
        return _lev(a, b) / len(a)

    return {"cer": rate(r, h), "wer": rate(r.split(), h.split())}


def word_diff(old, new):
    a, b = old.split(), new.split()
    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
    left, right = [], []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        aseg = html.escape(" ".join(a[i1:i2]))
        bseg = html.escape(" ".join(b[j1:j2]))
        if tag == "equal":
            left.append(aseg); right.append(bseg)
        elif tag == "replace":
            left.append(f'<span class="d">{aseg}</span>'); right.append(f'<span class="i">{bseg}</span>')
        elif tag == "delete":
            left.append(f'<span class="d">{aseg}</span>')
        elif tag == "insert":
            right.append(f'<span class="i">{bseg}</span>')
    return " ".join(left), " ".join(right)


# все картинки — по каждой сделаем одно предсказание
all_images = sorted(known_images, key=nat_key)

# для сравнения: старый ОДИНОЧНЫЙ json на картинку (сочетания старой модели к нашей неприменимы)
singles = {}
for fn in sorted(os.listdir(OCR_DIR)):
    if not fn.endswith(".json"):
        continue
    p = parse_doc(fn)
    if p and len(p) == 1 and p[0] in known_images:
        singles.setdefault(p[0], os.path.join(OCR_DIR, fn))

print(f"{len(all_images)} картинок; из них {len(singles)} с одиночным json старой модели для сравнения")

In [ ]:
from ap_res_cv_htr_entrypoint_protos.client import EntrypointClientAsync

entry_client = EntrypointClientAsync(ENDPOINT)

In [ ]:
# предсказания новой модели по каждой картинке: строки + полигоны детектора (кэш в CACHE_FILE / COORDS_FILE)
new_preds, new_coords = {}, {}
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, encoding="utf-8") as f:
        new_preds = json.load(f)
if os.path.exists(COORDS_FILE):
    with open(COORDS_FILE, encoding="utf-8") as f:
        new_coords = json.load(f)

for name in all_images:
    if name in new_preds and name in new_coords:   # оба закэшированы — пропускаем
        continue
    with open(os.path.join(IMAGES_DIR, name), "rb") as f:
        image_bytes = f.read()
    lines, coords = await entry_client.get_line_texts(image_bytes)
    new_preds[name] = lines
    new_coords[name] = [[[float(x), float(y)] for x, y in poly] for poly in coords]
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(new_preds, f, ensure_ascii=False, indent=1)
    with open(COORDS_FILE, "w", encoding="utf-8") as f:
        json.dump(new_coords, f, ensure_ascii=False)
    print(f"{name}: {len(lines)} строк, {len(coords)} боксов")
print("готово,", len(new_preds), "предсказаний")

In [ ]:
# сохраняем новую модель ПО ОДНОМУ ФАЙЛУ НА КАРТИНКУ, формат как у старой:
#   имя  = <DATE>_<картинка>.json
#   тело = {"id", "text"}, text = "Изображение 1:\n<строки через \n>"
os.makedirs(OUT_DIR, exist_ok=True)
for img in all_images:
    text = "\n".join(new_preds.get(img, []))
    out = {"id": str(uuid.uuid4()), "text": f"Изображение 1:\n{text}"}
    with open(os.path.join(OUT_DIR, f"{DATE}_{img}.json"), "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
print(f"сохранено {len(all_images)} файлов в {OUT_DIR}/ (по одному на картинку)")

In [ ]:
# сводка по картинкам: где новая и старая расходятся сильнее (маркеры "Изображение N:" не учитываем)
rows = []
for img, oldpath in singles.items():
    old = strip_prefix(load_text(oldpath))
    new = strip_prefix(load_text(os.path.join(OUT_DIR, f"{DATE}_{img}.json")))
    m = metrics(old, new)
    rows.append((img, m["cer"], m["wer"], len(new_preds.get(img, [])), len(norm(old)), len(norm(new))))
rows.sort(key=lambda r: (r[1] != r[1], -r[1]))  # по убыванию CER, nan в конец
order = [r[0] for r in rows]

hdr = ["картинка", "CER", "WER", "строк (new)", "|old|", "|new|"]
tbl = "<table style='border-collapse:collapse'><tr>" + "".join(
    f"<th style='border:1px solid #ccc;padding:4px'>{h}</th>" for h in hdr) + "</tr>"
for img, cer, wer, nl, lo, ln in rows:
    bg = "#ffecec" if cer == cer and cer > 0.3 else "#fff"
    cells = [img, f"{cer:.3f}", f"{wer:.3f}", nl, lo, ln]
    tbl += f"<tr style='background:{bg}'>" + "".join(
        f"<td style='border:1px solid #ccc;padding:4px'>{c}</td>" for c in cells) + "</tr>"
tbl += "</table>"
display(HTML(tbl))

In [ ]:
import io, base64
from PIL import Image as PILImage, ImageDraw, ImageFont

# шрифт для подписи номеров боксов (Linux -> DejaVu, Windows -> arial, иначе встроенный)
_FONT = None
for _cand in ("DejaVuSans.ttf", "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", "arial.ttf"):
    try:
        _FONT = ImageFont.truetype(_cand, 24); break
    except OSError:
        continue
if _FONT is None:
    _FONT = ImageFont.load_default()


def draw_overlay(img_path, polys):
    im = PILImage.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(im)
    for i, poly in enumerate(polys):
        pts = [tuple(p) for p in poly]
        if len(pts) < 2:
            continue
        draw.polygon(pts, outline=(255, 0, 0), width=3)
        x, y = pts[0]
        draw.text((max(0, x - 35), y), str(i), fill=(255, 0, 0), font=_FONT)
    return im


def to_datauri(im, max_w=MAX_IMG_WIDTH):
    if im.width > max_w:
        im = im.resize((max_w, int(im.height * max_w / im.width)))
    buf = io.BytesIO()
    im.convert("RGB").save(buf, format="JPEG", quality=80)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()


CSS = """<style>
.rep{font-family:sans-serif;color:#111}
.rep .cmp{margin:6px 0 34px;border-top:2px solid #ddd;padding-top:12px}
.rep .cmp h3{margin:4px 0}
.rep .imgs{display:flex;gap:12px;flex-wrap:wrap;margin:8px 0}
.rep .imgs figure{margin:0;max-width:49%}
.rep .imgs img{max-width:100%;border:1px solid #ccc}
.rep .imgs figcaption{font-size:12px;color:#666;margin-bottom:2px}
.rep .cmptab{border-collapse:collapse;width:100%;table-layout:fixed}
.rep .cmptab td{vertical-align:top;width:50%;padding:8px;border:1px solid #ccc;white-space:pre-wrap;font:13px/1.5 monospace}
.rep .cmptab th{padding:4px;background:#f0f0f0}
.rep .d{background:#ffd6d6}
.rep .i{background:#d6ffd6}
.rep .meta{color:#444;font-size:13px;margin:2px 0 6px}
</style>"""

sections = []
for img in order:
    old = strip_prefix(load_text(singles[img]))
    new = strip_prefix(load_text(os.path.join(OUT_DIR, f"{DATE}_{img}.json")))
    m = metrics(old, new)
    left, right = word_diff(old, new)
    polys = new_coords.get(img, [])
    orig_uri = to_datauri(PILImage.open(os.path.join(IMAGES_DIR, img)))
    ov_uri = to_datauri(draw_overlay(os.path.join(IMAGES_DIR, img), polys))
    sections.append(
        f'<div class="cmp"><h3>{img} &mdash; CER={m["cer"]:.3f}, WER={m["wer"]:.3f}</h3>'
        f'<div class="imgs">'
        f'<figure><figcaption>оригинал</figcaption><img src="{orig_uri}"></figure>'
        f'<figure><figcaption>детекция: {len(polys)} строк</figcaption><img src="{ov_uri}"></figure>'
        f'</div>'
        f'<div class="meta">(<span class="d">красное</span> — только в старой, '
        f'<span class="i">зелёное</span> — только в новой)</div>'
        f'<table class="cmptab"><tr><th>Старая (VLM)</th><th>Новая (HTR)</th></tr>'
        f'<tr><td>{left}</td><td>{right}</td></tr></table></div>'
    )

body = f'<div class="rep"><h1>Сравнение OCR: новая (HTR) vs старая (VLM) — {DATE}</h1>' \
       f'{globals().get("tbl", "")}{"".join(sections)}</div>'

# самодостаточный html-файл
report = f"<!doctype html><html><head><meta charset='utf-8'><title>OCR report {DATE}</title>{CSS}</head><body>{body}</body></html>"
with open(REPORT_FILE, "w", encoding="utf-8") as f:
    f.write(report)
print(f"отчёт: {os.path.abspath(REPORT_FILE)}  ({len(report)//1024} KB)")

# и то же самое прямо в ноутбуке
display(HTML(CSS + body))

In [ ]:
# ДИАГНОСТИКА координат детектора: в каком пространстве coords и укладываются ли в размер картинки
dbg = "image_15.jpg"   # любая картинка
im = PILImage.open(os.path.join(IMAGES_DIR, dbg))
polys = new_coords.get(dbg, [])
xs = [p[0] for poly in polys for p in poly]
ys = [p[1] for poly in polys for p in poly]
print("картинка:", dbg, "| размер (W x H):", im.size)
print("полигонов:", len(polys), "| точек в первом полигоне:", len(polys[0]) if polys else 0)
if xs:
    print(f"x: min={min(xs):.0f}  max={max(xs):.0f}   (должно укладываться в 0..{im.width})")
    print(f"y: min={min(ys):.0f}  max={max(ys):.0f}   (должно укладываться в 0..{im.height})")
    print("первый полигон:", polys[0])
# вывод: если max(x) > W или max(y) > H — координаты в другом масштабе (баг постпроцессинга сервиса),
#        а не в ноутбуке; если в пределах — детектор реально так задетектил.